## Import Library

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

pd.set_option('display.max_columns', None)

## Task TODO
0. Setup & constants
1. Load challenger models + metrics (from models/credit_risk/)
2. Challenger scoreboard  →  (Task: Model Development — challengers)
3. Model Monitoring: Gini, KS, PSI  →  (Task: Model Monitoring)
4. Income estimation model (XGBoost regressor, MAPE)  →  (Task: Model Development — income)
5. Propensity model (XGBoost, Average Precision)  →  (Task: Model Development — propensity)
6. Backtesting third-party vendor data  →  (Task: Backtesting)
7. Portfolio evaluation (NPF, FPD30)  →  (Task: Model Evaluation)
8. Automation hooks (report generation, DB monitor bot)  →  (Task: Automation)
9. Summary table of what runs on existing artifacts vs what needs extra data

## Setup database, Feature, ML Models

In [2]:
BASE_PATH    = Path.cwd().resolve().parents[0]           
PROJECT      = BASE_PATH / "credit_risk_production"
MODEL_DIR    = PROJECT / "models" / "credit_risk"
ML_DIR       = MODEL_DIR / "ml_credit_risk"
META_PATH    = MODEL_DIR / "metadata_credit_risk" / "metadata.json"
METRICS_PATH = MODEL_DIR / "metrics_credit_risk" / "model_metrics.csv"
PARAMS_PATH  = MODEL_DIR / "params_credit_risk"  / "best_parameters.json"
CM_PATH      = MODEL_DIR / "confusion_matrices.npz"
PARQUET_PATH = PROJECT / "database" / "data" / "merged_credit_risk_data.parquet"

# ---- Target / feature spec (from metadata.json) ----
TARGET = "Approved_Flag"
ID_COL = "PROSPECTID"

with open(META_PATH) as f:
    META = json.load(f)

FEATURES = META["feature_columns"]
CLASS_LABELS = META["class_labels"]

print(f"repo:       {BASE_PATH}")
print(f"model dir:  {MODEL_DIR}")
print(f"features:   {len(FEATURES)}")
print(f"classes:    {CLASS_LABELS}")

repo:       /Users/miftahhadiyannoor/Documents/credit_risk
model dir:  /Users/miftahhadiyannoor/Documents/credit_risk/credit_risk_production/models/credit_risk
features:   47
classes:    [0, 1, 2, 3]


## Load ML models + metrics

In [3]:
# Load ML models
def load_challenger_models() -> dict:
    """Load the six saved estimators keyed by their model name."""
    out = {}
    for name, fname in META["model_files"].items():
        path = ML_DIR / fname

        if not path.exists():
            print(f"⚠️ Missing: {path}")
            continue
        out[name] = joblib.load(path)
    return out

CHALLENGERS = load_challenger_models()
print(f"Load models: {list(CHALLENGERS.keys())}")

# Load metrics
metrics_df = pd.read_csv(METRICS_PATH)
metrics_df = metrics_df.sort_values("F1 Score", ascending=False).reset_index(drop=True)
print(f"\nMetrics for {len(metrics_df)} models:")
display(metrics_df)

with open(PARAMS_PATH) as f:
    best_params = json.load(f)
print(f"\nBest params per model:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# Load confusion matrices
cms = np.load(CM_PATH, allow_pickle=True)
print("\nconfusion matrices stored for:", list(cms.files))

Load models: ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost', 'K-Nearest Neighbors', 'Decision Tree']

Metrics for 6 models:


,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Gradient Boosting,0.995325,0.995450,0.995325,0.995314,0.999937
1,XGBoost,0.995228,0.995334,0.995228,0.995217,0.999917
2,Decision Tree,0.994644,0.994660,0.994644,0.994638,0.999067
3,Random Forest,0.989774,0.990072,0.989774,0.989735,0.999792
4,Logistic Regression,0.968056,0.968218,0.968056,0.967177,0.988453
5,K-Nearest Neighbors,0.775808,0.762119,0.775808,0.755816,0.898257



Best params per model:
  Logistic Regression: {'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000, 'C': 100}
  Random Forest: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 30}
  Gradient Boosting: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 3, 'learning_rate': 0.01}
  XGBoost: {'subsample': 0.9, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
  K-Nearest Neighbors: {'weights': 'distance', 'n_neighbors': 9, 'metric': 'manhattan'}
  Decision Tree: {'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 10}

confusion matrices stored for: ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost', 'K-Nearest Neighbors', 'Decision Tree']


## Challenger scoreboard (Task: Model Development)

In [4]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = pd.read_parquet(PARQUET_PATH, columns=[ID_COL, TARGET] + FEATURES)

# Encode categorical features usin g the same mapping as used in training
le_map = {}
for col in df.select_dtypes(include=["object", "category"]).columns:
    if col == TARGET:
        continue
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_map[col] = le

# Encode target
target_le = LabelEncoder()
y_all = target_le.fit_transform(df[TARGET].astype(str))
X_all = df[FEATURES].copy()

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

# Score every challenger on the same holdout
rows = []
for name, model in CHALLENGERS.items():
    proba = model.predict_proba(X_test)
    preds = proba.argmax(axis=1)
    rows.append({
        "Model": name,
        "ROC_AUC": roc_auc_score(y_test, proba, multi_class="ovr", average="weighted"),
        "F1 Score": metrics_df.loc[metrics_df["Model"] == name, "F1 Score"].values[0],
        "Accuracy": metrics_df.loc[metrics_df["Model"] == name, "Accuracy"].values[0],
        "Precision": metrics_df.loc[metrics_df["Model"] == name, "Precision"].values[0],
        "Recall": metrics_df.loc[metrics_df["Model"] == name, "Recall"].values[0]
    })

scoreboard = pd.DataFrame(rows).sort_values("F1 Score", ascending=False).reset_index(drop=True)
display(scoreboard)

# Champin the actually push to production model
CHAMPION_NAME = scoreboard.iloc[0]["Model"]
CHAMPION_MODEL = CHALLENGERS[CHAMPION_NAME]
print(f"\n🏆 champion Model: {CHAMPION_NAME}")

,Model,ROC_AUC,F1 Score,Accuracy,Precision,Recall
0,Gradient Boosting,0.520794,0.995314,0.995325,0.995450,0.995325
1,XGBoost,0.442547,0.995217,0.995228,0.995334,0.995228
2,Decision Tree,0.499711,0.994638,0.994644,0.994660,0.994644
3,Random Forest,0.590884,0.989735,0.989774,0.990072,0.989774
4,Logistic Regression,0.500049,0.967177,0.968056,0.968218,0.968056
5,K-Nearest Neighbors,0.615646,0.755816,0.775808,0.762119,0.775808



🏆 champion Model: Gradient Boosting


## Model Monitoring: Gini, KS, PSI (Task: Model Monitoring)

### Implement the monitoring metrics:
- #### Important caveat: Gini and KS need a binary target -> We'll do it one-vs-rest for the "worst" class, which is the standard way to compute Gini/KS on a multiclass credit model.
    

In [ ]:
from sklearn.metrics import roc_curve

def gini(y_true_bin: np.ndarray, y_score: np.ndarray) -> float:
    """Gini = 2*AUC - 1 (binary only)"""
    auc = roc_auc_score(y_true_bin, y_score)
    return 2 * auc - 1

def ks(y_true_bin: np.ndarray, y_score: np.ndarray) -> float:
    """Kolmogorov-Smirnov statistic: max |CDF_good - CDF_bad"""
    fpr, tpr, _ = roc_curve(y_true_bin, y_score)
    return float(np.max(np.abs(tpr - fpr)))

def psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    """Population Stability Index between two 1-D score distribution.
    PSI < 0.10 -> stable
    0.10 - 0.25 -> moderate shift
    > 0.25 -> significant shift (Investigate)
    """
    # Build bins from the expected distribution
    breakpoints = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))

    if len(breakpoints) < 3:
        return 0.0 

    e_counts, _ = np.histogram(expected, bins=breakpoints)
    a_counts, _ = np.histogram(actual, bins=breakpoints)

    e_pct = np.clip(e_counts / max(e_counts.sum(), 1), 1e-6, None)
    a_pct = np.clip(a_counts / max(a_counts.sum(), 1), 1e-6, None)
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))

# --- Gini / KS on the champion model --- 
CLASS_OF_INTEREST = CLASS_LABELS[1] # Assuming the positive class is the second label
y_bin_test = (y_test == CLASS_OF_INTEREST).astype(int)
champion_proba = CHAMPION_MODEL.predict_proba(X_test)[:, CLASS_OF_INTEREST]

print(f"Champion: {CHAMPION_NAME}")
print(f"Class of interest: {CLASS_OF_INTEREST}")
print(f"Gini: {gini(y_bin_test, champion_proba):.4f}")
print(f"KS: {ks(y_bin_test, champion_proba):.4f}")



Champion: Gradient Boosting
Class of interest: 1
Gini: 0.1776
KS: 0.1970
